# 🧬 Project: Bacterial Single-Cell RNA-Seq Analysis (microSPLiT)
## 📓 Notebook 01: Data Loading, AnnData Construction & Quality Control

### 📌 Overview
In this notebook, we begin the downstream analysis of single-cell RNA-seq data from *Bacillus subtilis* (Dataset: **GSE151940**), generated using the **microSPLiT** technique. We will focus on the main growth-course experiment (`GSM4594095`) tracking bacterial cells across 10 different Optical Densities (OD0.5 to OD6.0).

### 🎯 Objectives for this step:
1. Load the raw count matrix, gene names, and cell metadata using `Pandas`.
2. Construct a unified `AnnData` object compatible with `Scanpy`.
3. Calculate Quality Control (QC) metrics specific to bacterial cells and perform initial filtering.

In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [6]:
# Load cell metadata WITH header=None so we don't lose the first cell
metadata = pd.read_csv('data/GSM4594095_mRNA_M14_ODannotation.csv.gz', header=None, names=['OD_stage'])
print(f"Metadata shape: {metadata.shape}")

Metadata shape: (25446, 1)


In [5]:
# Load the main expression matrix
# It is ALREADY in (Cells x Genes) format, so NO NEED for .T
counts = pd.read_csv('data/GSM4594095_mRNA_M14_200.csv.gz', header=None)
print(f"Counts Matrix shape (cells x genes): {counts.shape}")

Counts Matrix shape (cells x genes): (25446, 3769)


### 🏗️ Step 2: Constructing the AnnData Object

`Scanpy` relies on the `AnnData` (Annotated Data) object to store single-cell datasets. It acts as a unified container mapping three main components:
* **`X`**: The core expression matrix (cells $\times$ genes).
* **`obs` (Observations)**: A dataframe storing cell-level metadata (e.g., OD stages).
* **`var` (Variables)**: A dataframe storing gene-level metadata (e.g., gene symbols).

Here, we instantiate the `AnnData` object, align our metadata, and assign unique identifiers (barcodes) to our bacterial cells.

In [7]:
# 1. Initialize the AnnData object with the count matrix
adata = sc.AnnData(X=counts.values)

# 2. Add cell metadata to the .obs (Observations) attribute
adata.obs['OD_stage'] = metadata['OD_stage'].values

# 3. Add gene names to the .var (Variables) attribute and set them as index
adata.var_names = genes['gene_symbols'].values
adata.var['gene_name'] = genes['gene_symbols'].values

# 4. Generate unique names (barcodes) for the cells
adata.obs_names = [f"cell_{i}" for i in range(adata.n_obs)]

# 5. Make gene names unique (in case there are duplicates in the raw data)
adata.var_names_make_unique()

# Print the final object to see its structure
print(adata)

AnnData object with n_obs × n_vars = 25446 × 3769
    obs: 'OD_stage'
    var: 'gene_name'


### 💾 Step 3: Saving the AnnData Object to Disk

Currently, the `AnnData` object exists only in memory (RAM). To avoid reprocessing the raw text files in the future, we will save this structured object to disk as an `.h5ad` file. This format is highly optimized for single-cell data and can be easily loaded into any Python environment.

In [8]:
# Define the path and filename for our new dataset
output_file = 'data/bacterial_microSPLiT_raw.h5ad'

# Save the AnnData object
adata.write(output_file)

print(f"✅ Success! The AnnData object is securely saved to: {output_file}")

✅ Success! The AnnData object is securely saved to: data/bacterial_microSPLiT_raw.h5ad
